# Advanced Exercise — Graph-based Recommendations (Solution)

In this exercise, you will:
1) Build a bipartite user–item graph from MovieLens (ml-20m)
2) Use a pre-trained LightGCN to obtain embeddings
3) Extract user and item embeddings
4) Implement an embeddings-based search (item–item and personalized Top-K)

Style note: This follows the format of the Essentials exercises. Later, you'll redact parts into TODOs for students.

## Setup
- Run from repo root so paths resolve (`./data/...`).
- Data: `./data/ml-20m` (already in this repo).
- Pre-trained embeddings: produced by [`02 - Advanced/lightgcn/train.py`](lightgcn/train.py).

In [10]:
import os, sys, math, random, json
import numpy as np
import pandas as pd
from scipy import sparse
import torch

DATA_ROOT = "../data/ml-20m"
# change if you saved elsewhere
MODEL_PATH = "./lightgcn/lightgcn-ml20m-emb-64.pt"

# For item titles
movies = pd.read_csv(os.path.join(DATA_ROOT, "movies.csv"))
ratings = pd.read_csv(os.path.join(DATA_ROOT, "ratings.csv"), usecols=["userId","movieId","rating"])

## Phase 1: Graph Construction (PyTorch Geometric)
We construct a bipartite graph from ratings >= 4.0 as implicit positive edges.
- Build contiguous ID mappings for users and items
- Prepare PyG `edge_index` (if available) and a SciPy CSR for reference
- Keep this phase fast by allowing an optional limit on users/items

In [20]:
MIN_RATING = 4.0
LIMIT_USERS = None  # e.g., 50_000 for faster debug
LIMIT_ITEMS = None

pos = ratings[ratings["rating"] >= MIN_RATING].copy()
if LIMIT_USERS is not None:
    keep_u = sorted(pos["userId"].unique())[:int(LIMIT_USERS)]
    pos = pos[pos["userId"].isin(keep_u)]
if LIMIT_ITEMS is not None:
    keep_i = sorted(pos["movieId"].unique())[:int(LIMIT_ITEMS)]
    pos = pos[pos["movieId"].isin(keep_i)]

u_raw = sorted(pos["userId"].unique())
i_raw = sorted(pos["movieId"].unique())
u2idx = {u:i for i,u in enumerate(u_raw)}
i2idx = {m:j for j,m in enumerate(i_raw)}
idx2u = u_raw
idx2i = i_raw
U, I = len(u_raw), len(i_raw)
print(f"Users: {U}, Items: {I}, Positives: {len(pos)}")

rows = np.fromiter((u2idx[u] for u in pos["userId"].values), dtype=np.int64, count=len(pos))
cols = np.fromiter((i2idx[m] for m in pos["movieId"].values), dtype=np.int64, count=len(pos))
data = np.ones(len(pos), dtype=np.float32)
R = sparse.coo_matrix((data, (rows, cols)), shape=(U, I)).tocsr()

# Optional: build PyG edge_index if torch_geometric is installed
edge_index = None
try:
    import torch_geometric
    import torch
    ui = np.vstack([rows, cols + U])  # share index space [0..U+I)
    edge_index = torch.tensor(ui, dtype=torch.long)
    # Make it undirected (LightGCN expects both directions)
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    print("PyG available — edge_index built:", tuple(edge_index.shape))
except Exception as e:
    print("PyG not available; continuing with SciPy CSR only.")

Users: 138287, Items: 20720, Positives: 9995410
PyG available — edge_index built: (2, 19990820)


## Phase 2: GNN Training (pre-trained LightGCN)
We use a pre-trained LightGCN model from the helper package under [`02 - Advanced/lightgcn/train.py`](lightgcn/train.py).

However, if you really want you can train the model yourself (although it shouldn't be necessary). To do so, run (careful with the paths!):
  - `python "02 - Advanced/lightgcn/train.py" --data-root ./data/ml-20m --epochs 3 --steps-per-epoch 300 --alpha-mode learnable --edge-dropout 0.1 --neg-sampling popularity --out ./data/lightgcn_ml20m.pt`

In [16]:
payload = torch.load(MODEL_PATH, weights_only=False, map_location="cpu")
print({k:type(v).__name__ for k,v in payload.items() if k not in ("E0","Z","u2idx","i2idx","idx2u","idx2i")})
Z = payload["Z"]; E0 = payload["E0"]  # tensors
U_payload = int(payload["num_users"]) ; I_payload = int(payload["num_items"]) 
assert U == U_payload and I == I_payload, "ID mapping mismatch vs payload. Ensure MIN_RATING and limits match training."
print("Embeddings:", tuple(Z.shape), "Base:", tuple(E0.shape))

{'config': 'dict', 'num_users': 'int', 'num_items': 'int'}
Embeddings: (159007, 64) Base: (159007, 64)


## Phase 3: Embeddings Extraction
Split node embeddings into users and items, and (optionally) L2-normalize the item embeddings for cosine search.

In [17]:
U_emb = Z[:U].contiguous()
I_emb = Z[U:].contiguous()
print("U_emb:", tuple(U_emb.shape), "I_emb:", tuple(I_emb.shape))

from lightgcn.inference import l2_normalize
I_emb_norm = l2_normalize(I_emb)

U_emb: (138287, 64) I_emb: (20720, 64)


## Phase 4: Search System (embeddings)
- Item–Item: cosine similarity over item embeddings by title substring
- Personalized Top-K: dot product between user and item embeddings
- Optional: export Top-K CSV for BLU12 evaluation

In [18]:
from lightgcn.inference import similar_items_by_title, recommend_for_user_topk, write_topk_csv

# Build raw user -> seen items map for filtering (rating >= MIN_RATING)
user_pos_raw = {}
for uid, mid in pos[["userId","movieId"]].itertuples(index=False):
    user_pos_raw.setdefault(int(uid), set()).add(int(mid))

# Quick demo — similar items by title
demo = similar_items_by_title(
    query="Toy Story",
    I_emb=I_emb_norm, movies_df=movies, idx2i=idx2i, i2idx={m:i for i,m in enumerate(idx2i)},
    topk=10, normalize=True,
)
for mid, title, sim in demo:
    print(f"{title} [{mid}]  sim={sim:.3f}")

# Quick demo — personalized Top-K for one user
sample_user = u_raw[0]
recs = recommend_for_user_topk(
    user_raw_id=sample_user,
    U_emb=U_emb, I_emb=I_emb,
    u2idx=u2idx, idx2i=idx2i,
    topk=10, exclude_seen=True, user_pos_raw=user_pos_raw,
)
titles = movies.set_index("movieId")["title"]
for mid, score in recs:
    print(f"{titles.get(mid, mid)} [{mid}]  score={score:.4f}")

Toy Story (1995) [1]  sim=1.000
Blown Away (1994) [423]  sim=1.000
Pinocchio (1940) [596]  sim=1.000
Hard Rain (1998) [1752]  sim=1.000
Lamerica (1994) [53]  sim=1.000
Before the Rain (Pred dozhdot) (1994) [214]  sim=1.000
Paulie (1998) [1806]  sim=1.000
Willy Wonka & the Chocolate Factory (1971) [1073]  sim=1.000
Red Firecracker, Green Firecracker (Pao Da Shuang Deng) (1994) [309]  sim=1.000
Adventures of Priscilla, Queen of the Desert, The (1994) [345]  sim=1.000
Silence of the Lambs, The (1991) [593]  score=12.9855
Forrest Gump (1994) [356]  score=12.5238
Schindler's List (1993) [527]  score=11.7703
Usual Suspects, The (1995) [50]  score=11.2389
Braveheart (1995) [110]  score=11.1581
Fugitive, The (1993) [457]  score=11.1124
Apollo 13 (1995) [150]  score=10.8534
Matrix, The (1999) [2571]  score=10.7685
Toy Story (1995) [1]  score=10.7199
Godfather, The (1972) [858]  score=10.5328


### Optional: Export predictions for BLU12 evaluation
This writes `data/<pred_name>.csv` with one row per user and Top-K itemIds.
Then run: `python "01 - Essentials/BLU12 - Workflow/evaluation.py" <pred_name>`

In [ ]:
PRED_NAME = "lightgcn_ml20m_topk"
TOPK = 10
pred_path = f"../data/{PRED_NAME}.csv"

# For speed, you may restrict the user set; here we use all payload users
write_topk_csv(
    out_path=pred_path,
    user_ids=idx2u,  # raw userIds
    U_emb=U_emb, I_emb=I_emb,
    u2idx=u2idx, idx2i=idx2i,
    topk=TOPK, exclude_seen=True, user_pos_raw=user_pos_raw,
)
print("Wrote:", pred_path)
print("Evaluate:", "python \"01 - Essentials/BLU12 - Workflow/evaluation.py\"", PRED_NAME)

## Notes
- Ensure `MIN_RATING`, user/item limits and mappings match your training run.
- For very large user sets, export predictions for a subset to keep evaluation fast.
- You can switch from dot product to cosine by normalizing both U and I.